# help! — pi0.5 LoRA fine-tune on Colab

Fine-tunes pi0.5 (`pi05_base`, LoRA) on one help! task's expert demos (TASK_SPEC.md §2–3, §7), with upstream openpi
and `helpsim/openpi_help.py`. Run on an **A100** runtime (LoRA needs > 22.5 GB).

1. Set `TASK` below (`shelf` or `drawer`).
2. Run the cells in order. The smoke cell measures seconds per step, so you can size the full run before spending units on it.
3. Norm stats and checkpoints go to Drive (`ROOT`), so a disconnected runtime picks up again with the same cells.

The datasets are LeRobot datasets made by `scripts/to_lerobot.py` and uploaded to `DATA/lerobot/help/<task>_pi05`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

TASK = 'shelf'                      # 'shelf' or 'drawer'
DATA = '/content/drive/MyDrive/stanford/classes/cs238/data'
ROOT = f'{DATA}/pi05'               # norm stats (assets/) and checkpoints/
REPO = f'help/{TASK}_pi05'
OPENPI_COMMIT = '215abfb217dbac7d5f1273282331b9b1866c0479'   # helpsim/openpi_help.py OPENPI_COMMIT

assert os.path.isdir(f'{DATA}/lerobot/{REPO}'), f'dataset not on Drive: {DATA}/lerobot/{REPO}'
os.environ.update(TASK=TASK, DATA=DATA, ROOT=ROOT, REPO=REPO, OPENPI_COMMIT=OPENPI_COMMIT,
                  HF_LEROBOT_HOME='/content/lerobot', XLA_PYTHON_CLIENT_MEM_FRACTION='0.9')

Install upstream openpi (pinned) and this repo. About 5 minutes on a fresh runtime.

In [ ]:
%%bash
set -e
cd /content
[ -d openpi ] || git clone -q https://github.com/Physical-Intelligence/openpi.git
cd openpi && git fetch -q origin "$OPENPI_COMMIT" 2>/dev/null || true
git checkout -q "$OPENPI_COMMIT" && git submodule update -q --init --recursive
pip -q install uv
GIT_LFS_SKIP_SMUDGE=1 uv sync -q
cd /content
if [ -d help-your-robot ]; then git -C help-your-robot pull -q; else git clone -q https://github.com/DarinAnthony/help-your-robot.git; fi
echo installed

Copy the dataset to the runtime's local disk. Video decoding from the Drive mount is slow.

In [ ]:
%%bash
mkdir -p /content/lerobot/help
rsync -a "$DATA/lerobot/$REPO" /content/lerobot/help/
du -sh "/content/lerobot/$REPO"; python -c "import json; d=json.load(open('/content/lerobot/$REPO/meta/info.json')); print(d['total_episodes'], 'episodes,', d['total_frames'], 'frames')"

Normalization statistics (once per dataset; written to `ROOT/assets`).

In [ ]:
%%bash
cd /content/openpi
uv run python /content/help-your-robot/scripts/openpi_run.py norm-stats $TASK --repo $REPO --root $ROOT

**Smoke run.** 200 steps. Check that the loss falls, and note the seconds per step: a 10k-step run takes about
10,000 × that. On an A100 runtime, multiply the hours by its compute units per hour (shown in Colab's resources panel).

In [ ]:
%%bash
cd /content/openpi
uv run python /content/help-your-robot/scripts/openpi_run.py train $TASK --repo $REPO --root $ROOT \
    --exp smoke --steps 200 --log-interval 10 --overwrite 2>&1 | grep -v -i warning

**Full run.** `--resume` continues from the last checkpoint on Drive after a disconnect (a missing run starts fresh).
Set `STEPS` from the smoke run's timing.

In [ ]:
STEPS = 10_000
os.environ['STEPS'] = str(STEPS)

In [ ]:
%%bash
cd /content/openpi
uv run python /content/help-your-robot/scripts/openpi_run.py train $TASK --repo $REPO --root $ROOT \
    --exp run1 --steps $STEPS --resume 2>&1 | grep -v -i warning

**Serve the policy** for evaluation in the sim on the Mac (TASK_SPEC.md §7). Starts openpi's websocket server on
port 8000. The tunnel to reach it from the Mac is set up separately.

In [ ]:
%%bash
cd /content/openpi
uv run python /content/help-your-robot/scripts/openpi_run.py serve $TASK --repo $REPO --root $ROOT --exp run1